# Issue 006 Prompt #1 — Pipeline A VideoMAE Scaffold

**Status:** scaffold only. Data wiring, GPU recording, tiny-slice overfit, evaluation-harness plumbing. **Not** in this step: full fine-tune, real validation, scheduler, or weights pushed anywhere.

This notebook reads **already-computed crop shards** from Drive artefacts and **never re-runs** raw-frame staging, perception, or cropping. If those shards are missing, the notebook fails loud with a clear "run `colab/000_full_pipeline.ipynb` first" message.

Reuses the existing setup/layout modules instead of duplicating configuration:

- `colab/setup.py` (Drive layout, env lock, run log)
- `colab/data_mode.py` (LOCAL / DRIVE resolution)
- `pipeline_a/load_crops.py` (the Issue 005 Step 1 crop loader)
- `cropping/shard_writer.py` (read-only shard utilities — *not* the runner)
- `data/urfd_labels.py` (`label_window_from_crop_meta` — the supported real-crop entry point)
- `evaluation/metrics/classification.py` (the Issue 004 Step 2 metric bundle)

### Hard rules carried forward from Issues 001–005

- Detector was `yolo26m`, tracker was `bytetrack.yaml`. **Out of scope** for this notebook — Issue 002 produced the shards.
- Crop shards are 32 × 224 × 224 (URFD cam0 only), ImageNet-normalised by the loader. This notebook does NOT re-normalise.
- Train / validation split is **by source clip**, never by window.
- Frozen unseen tier is **inaccessible** from this scaffold — `ExecutionContext.EVALUATION` denies frozen access by design.
- The Hugging Face video-classification tutorial is an **API reference only** — its accuracy-only metric, `.avi` data pipeline, normalisation, and Hub-push flow are NOT copied.

### Why this is only Prompt #1

Per the Issue 006 spec, this notebook proves the wiring (model loads, dataset produces tensors of the right shape, loss can collapse on a tiny slice, eval plumbing produces real numbers). A long fine-tune run is a separate prompt.

## 1. Mount Drive + resolve layout

**LOCAL mode** is the default — datasets live on the Colab local disk during staging / perception / cropping. This scaffold reads only from Drive artefacts (shards, labels, logs), so Drive is the only thing that needs to be mounted.

Override via `FALL_DETECTION_DATA_MODE=local|drive` or by passing a different root. The notebook never stages anything from raw frames, so the dataset root is informational here.

In [ ]:
import os, pathlib, sys

# Mount Drive (or skip if not on Colab — local dev still works because the
# notebooks read from FALL_DETECTION_PROJECT_ROOT, not from /content/drive).
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not on Colab — Drive mount skipped. Set FALL_DETECTION_PROJECT_ROOT if running outside Colab.")

PROJECT_ROOT = pathlib.Path(os.environ.get("FALL_DETECTION_PROJECT_ROOT", "/content/fall_detection"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from colab.data_mode import DataMode, resolve_data_layout, describe_layout

DATA_MODE = DataMode(os.environ.get("FALL_DETECTION_DATA_MODE", "local").lower())
layout = resolve_data_layout(mode=DATA_MODE)

print(f"Data mode        : {layout.mode.value}")
print(f"Dataset root     : {layout.dataset_root}")
print(f"Artefact root    : {layout.artifact_root}")
print(f"Layout summary   : {describe_layout(layout)}")

# Resolved artefact paths this notebook writes to.
CROPS_ROOT = layout.artifacts / "crops"
CSV_LABEL_DIR = layout.artifact_root / "artifacts" / "urfd_labels"
CKPT_DIR = layout.artifact_root / "checkpoints" / "pipeline_a"
METRICS_DIR = layout.artifact_root / "metrics" / "pipeline_a"
for d in (CROPS_ROOT, CSV_LABEL_DIR, CKPT_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Crops root       : {CROPS_ROOT}")
print(f"CSV label dir    : {CSV_LABEL_DIR}")
print(f"Checkpoint dir   : {CKPT_DIR}")
print(f"Metrics dir      : {METRICS_DIR}")

## 1a. GPU / environment recording

The Issue 006 carry-forwards call out: capability-detect fp16 / bf16 before training, record actual GPU + VRAM + CUDA + torch, and pin the run with a JSON run-log sidecar so Issue 015 (real-time opt) and Issue 004 (eval harness) have stable provenance.

Pure stdlib + torch / subprocess — no ultralytics, no transformers (those imports land in §3 where they're needed).

In [ ]:
import json, platform, shutil, subprocess
from datetime import datetime, timezone

import torch

def _query_nvidia_smi():
    """Best-effort GPU name + VRAM (GiB) via nvidia-smi."""
    if shutil.which("nvidia-smi") is None:
        return None, None
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total",
             "--format=csv,noheader,nounits"],
            text=True, timeout=10).strip()
    except Exception:
        return None, None
    if not out:
        return None, None
    first = out.splitlines()[0]
    parts = [p.strip() for p in first.split(",")]
    if len(parts) < 2:
        return parts[0], None
    try:
        return parts[0], round(float(parts[1]) / 1024.0, 2)
    except ValueError:
        return parts[0], None

gpu_name, vram_gib = _query_nvidia_smi()
torch_cuda = torch.version.cuda
bf16_supported = bool(
    torch.cuda.is_available() and torch.cuda.is_bf16_supported()
) if hasattr(torch.cuda, "is_bf16_supported") else False
fp16_supported = bool(torch.cuda.is_available())

env_record = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "gpu_name": gpu_name,
    "gpu_vram_gib": vram_gib,
    "torch_version": torch.__version__,
    "torch_cuda_version": torch_cuda,
    "torch_cuda_available": bool(torch.cuda.is_available()),
    "bf16_supported": bf16_supported,
    "fp16_supported": fp16_supported,
    "python_version": platform.python_version(),
    "data_mode": layout.mode.value,
    "crops_root": str(CROPS_ROOT),
    "csv_label_dir": str(CSV_LABEL_DIR),
}

print("Recorded environment:")
for k, v in env_record.items():
    print(f"  {k:<24} = {v}")

# Persist the run-log sidecar so Issue 015 / Issue 004 can correlate.
RUN_LOG_PATH = METRICS_DIR / "006_pipeline_a_run_log.json"
RUN_LOG_PATH.write_text(json.dumps(env_record, indent=2, sort_keys=True), encoding="utf-8")
print(f"\nRun-log sidecar: {RUN_LOG_PATH}")

# Loud fail if no CUDA — VideoMAE on CPU is not the path we are testing here.
assert torch.cuda.is_available(), (
    "CUDA is not available. Issue 006 requires a Colab GPU runtime; "
    "switch the runtime type (T4 / L4 / A100) and re-run."
)

## 2. Crop-availability guard

This notebook reads `shard-*.tar` from `CROPS_ROOT`. The expected current cache shape (verified at Issue 005 close-out) is **70 shards, 240 windows, 41 fall, 199 no-fall**. If the cache is missing or inconsistent, the notebook **fails loud with a clear "run `colab/000_full_pipeline.ipynb` first"** — never silently rebuilds from raw frames.

The check is purely a guard: it reads `list_clip_keys` + one metadata sidecar per clip key. It does NOT load pixels.

In [ ]:
from pipeline_a import REQUIRED_METADATA_FIELDS
from cropping.shard_writer import read_shard, safe_member_name
from data.urfd_labels import (
    parse_urfd_csv_label_file,
    label_window_from_crop_meta,
    WINDOW_LABEL_FALL,
    WINDOW_LABEL_NO_FALL,
    WINDOW_LABEL_UNLABELED,
)

FALL_CSV_FILENAME = "urfall-cam0-falls.csv"
ADL_CSV_FILENAME = "urfall-cam0-adls.csv"

EXPECTED_SHARDS_MIN = 65
EXPECTED_SHARDS_MAX = 75
EXPECTED_WINDOWS_MIN = 220
EXPECTED_WINDOWS_MAX = 260
EXPECTED_UNIQUE_CLIPS_MIN = 65
EXPECTED_UNIQUE_CLIPS_MAX = 75

shard_paths = sorted(CROPS_ROOT.glob("shard-*.tar"))
if not shard_paths:
    raise SystemExit(
        f"No crop shards found under {CROPS_ROOT}.\n"
        f"Run colab/000_full_pipeline.ipynb first to produce shards, then re-run this notebook.\n"
        f"(That notebook handles Drive mounting, URFD staging, perception, and crop generation.)"
    )

# Walk every shard; for each clip_key, collect the window's frame_indices
# (one per sidecar, 0-based — the same indexing the pipeline_a loader
# uses) plus its source-clip label from the sidecar.
#
# meta["label"] is treated as SOURCE CLIP provenance only — it is the
# `clip_record.label` from the Issue 003 cropping runner, NOT a
# window-level fall/no_fall decision. Window-level labels come from
# label_window_from_crop_meta(...) below, never from meta["label"].
records = []  # list of dicts: clip_key, clip_id, source_label, frame_indices
for sp in shard_paths:
    res = read_shard(sp)
    for clip_key in res.manifest.get("clip_keys", []):
        safe = safe_member_name(clip_key)
        per_window_members = {
            name: meta for name, meta in res.metadata_members.items()
            if name.startswith(safe + "_") and name.endswith(".meta.json")
        }
        if not per_window_members:
            raise SystemExit(
                f"Shard {sp.name} clip {clip_key!r}: no metadata sidecars found.\n"
                f"Re-run Issue 003 (colab/000_full_pipeline.ipynb) to regenerate shards."
            )
        # Required-fields check on every sidecar (cheap dict membership).
        for name, meta in per_window_members.items():
            for f in REQUIRED_METADATA_FIELDS:
                if f not in meta:
                    raise SystemExit(
                        f"Shard {sp.name} sidecar {name!r}: missing required field {f!r}."
                    )
        # All sidecars for a window agree on provenance — take the first.
        first_meta = next(iter(per_window_members.values()))
        clip_id = first_meta["clip_id"]
        source_label = first_meta["label"]
        frame_indices = sorted(
            int(m["frame_index"]) for m in per_window_members.values() if "frame_index" in m
        )
        records.append({
            "clip_key": clip_key,
            "clip_id": clip_id,
            "source_label": source_label,
            "frame_indices": frame_indices,
        })

n_shards = len(shard_paths)
n_windows = len(records)
n_source_fall_windows = sum(1 for r in records if r["source_label"] == "fall")
n_source_nofall_windows = sum(1 for r in records if r["source_label"] == "no_fall")
unique_clip_ids = sorted({r["clip_id"] for r in records})

print(f"Shards on Drive        : {n_shards}    (expected {EXPECTED_SHARDS_MIN}..{EXPECTED_SHARDS_MAX})")
print(f"Windows on Drive       : {n_windows}    (expected {EXPECTED_WINDOWS_MIN}..{EXPECTED_WINDOWS_MAX})")
print(f"  source-label fall    : {n_source_fall_windows}    (windows from FALL clips — source-clip provenance, NOT window-level labels)")
print(f"  source-label no_fall : {n_source_nofall_windows}    (windows from ADL clips — source-clip provenance, NOT window-level labels)")
print(f"Unique source clips    : {len(unique_clip_ids)}    (expected {EXPECTED_UNIQUE_CLIPS_MIN}..{EXPECTED_UNIQUE_CLIPS_MAX})")

fail_msg = []
if not (EXPECTED_SHARDS_MIN <= n_shards <= EXPECTED_SHARDS_MAX):
    fail_msg.append(f"shard count {n_shards} is outside {EXPECTED_SHARDS_MIN}..{EXPECTED_SHARDS_MAX}.")
if not (EXPECTED_WINDOWS_MIN <= n_windows <= EXPECTED_WINDOWS_MAX):
    fail_msg.append(f"window count {n_windows} is outside {EXPECTED_WINDOWS_MIN}..{EXPECTED_WINDOWS_MAX}.")
if not (EXPECTED_UNIQUE_CLIPS_MIN <= len(unique_clip_ids) <= EXPECTED_UNIQUE_CLIPS_MAX):
    fail_msg.append(f"unique-clip count {len(unique_clip_ids)} is outside {EXPECTED_UNIQUE_CLIPS_MIN}..{EXPECTED_UNIQUE_CLIPS_MAX}.")
# Every clip_id must be URFD debug family with a recognised sequence.
for r in records:
    ci = r["clip_id"]
    if not (ci.startswith("urfd-debug-") and ci.endswith("-cam0-rgb")):
        fail_msg.append(f"unexpected clip_id shape: {ci!r} (every clip must be urfd-debug-<seq>-cam0-rgb)")
        break

# CSV presence.
fall_csv_path = CSV_LABEL_DIR / FALL_CSV_FILENAME
adl_csv_path = CSV_LABEL_DIR / ADL_CSV_FILENAME
for p in (fall_csv_path, adl_csv_path):
    if not p.is_file():
        raise SystemExit(
            f"URFD label CSV missing: {p}.\n"
            f"Run colab/000_full_pipeline.ipynb first to download the CSVs (university source)."
        )

# Stronger correct check: window-level labels via label_window_from_crop_meta.
# Cheap (240 windows × ~32 frame lookups, sub-second total) and catches
# the most common wiring regressions — wrong CSV, wrong offset, swapped
# fall/ADL directory — without requiring an exact count match. The exact
# window-level fall count depends on which windows span the falling/
# lying region vs. the pre-fall region of each fall clip, so we only
# check consistency invariants:
#
#   1. Every window resolves to one of {fall, no_fall, unlabeled}.
#   2. Sum of fall + no_fall + unlabeled equals total windows.
#   3. ADL-source (no_fall source label) windows never resolve to "fall"
#      via the CSV — that would be a CSV / offset wiring bug.
fall_csv = parse_urfd_csv_label_file(fall_csv_path)
adl_csv = parse_urfd_csv_label_file(adl_csv_path)

n_window_fall = 0
n_window_nofall = 0
n_window_unlabeled = 0
lookup_errors = []
for r in records:
    csv = fall_csv if "fall-" in r["clip_id"] else adl_csv
    try:
        label_str, _ = label_window_from_crop_meta(
            r["clip_id"], list(r["frame_indices"]), csv,
        )
    except Exception as exc:
        lookup_errors.append((r["clip_key"], f"{type(exc).__name__}: {exc}"))
        continue
    if label_str == WINDOW_LABEL_FALL:
        n_window_fall += 1
    elif label_str == WINDOW_LABEL_NO_FALL:
        n_window_nofall += 1
    elif label_str == WINDOW_LABEL_UNLABELED:
        n_window_unlabeled += 1
    else:
        lookup_errors.append((r["clip_key"], f"unexpected label_str {label_str!r}"))

total_labelled = n_window_fall + n_window_nofall + n_window_unlabeled
if total_labelled != n_windows:
    fail_msg.append(
        f"window-level labels cover {total_labelled} of {n_windows} windows; "
        f"every window must resolve to fall/no_fall/unlabeled."
    )

# Belt-and-braces: ADL-source windows must NEVER resolve to "fall".
adl_fall_mismatch = 0
for r in records:
    if r["source_label"] != "no_fall":
        continue
    label_str, _ = label_window_from_crop_meta(
        r["clip_id"], list(r["frame_indices"]), adl_csv,
    )
    if label_str == WINDOW_LABEL_FALL:
        adl_fall_mismatch += 1
if adl_fall_mismatch > 0:
    fail_msg.append(
        f"{adl_fall_mismatch} ADL-source windows resolved to 'fall' — "
        f"likely a CSV / offset wiring bug (ADL clips cannot contain a fall)."
    )

if lookup_errors:
    fail_msg.append(
        f"{len(lookup_errors)} window(s) had label-lookup errors. First: {lookup_errors[0]}"
    )

if fail_msg:
    raise SystemExit(
        "Crop cache shape inconsistent with the Issue 005 close-out.\n"
        "Failures:\n  - " + "\n  - ".join(fail_msg) + "\n\n"
        "Fix: run colab/000_full_pipeline.ipynb first (with FORCE_RECOMPUTE=True "
        "if the shards are stale), then re-run this notebook."
    )

print()
print("Window-level labels (via label_window_from_crop_meta — NOT meta['label']):")
print(f"  fall       : {n_window_fall}    (windows spanning the falling/lying region of a fall clip)")
print(f"  no_fall    : {n_window_nofall}    (pre-fall windows of fall clips + every ADL window)")
print(f"  unlabeled  : {n_window_unlabeled}    (windows with zero available CSV labels)")
print()
print("Crop-availability guard: PASSED.")

# Compatibility alias for later cells that consume `windows`.
# `records` carries the full per-window info (clip_key, clip_id,
# source_label, frame_indices); `windows` is the (clip_key,
# clip_id, source_label) tuple the §6 split cell expects. Defining
# it here — at the end of §2 — keeps §6 free of any reference to
# `records` (the §6 cell consumes only what it needs).
windows = [(r["clip_key"], r["clip_id"], r["source_label"]) for r in records]

## 3. VideoMAE model load

Loads `MCG-NJU/videomae-base` with a binary classification head. The Hugging Face video-classification tutorial is used only as an **API reference** for `VideoMAEForVideoClassification.from_pretrained` and the `label2id` / `id2label` contract — its tutorial-specific code (`.avi` pipeline, accuracy-only metric, Hub push, custom normalisation) is NOT used.

We print the model's configured frame count, image size, and tubelet size so §6 can reconcile the 32-frame crop to the model's expected frame count via uniform temporal subsampling.

In [ ]:
from transformers import VideoMAEForVideoClassification, VideoMAEConfig

MODEL_ID = "MCG-NJU/videomae-base"
LABEL2ID = {"no_fall": 0, "fall": 1}
ID2LABEL = {0: "no_fall", 1: "fall"}

model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

model_cfg: VideoMAEConfig = model.config
MODEL_T = int(model_cfg.num_frames)
MODEL_H = int(model_cfg.image_size)
MODEL_W = int(model_cfg.image_size)
TUBELET = int(model_cfg.tubelet_size)
PATCH = int(model_cfg.patch_size)
HIDDEN = int(model_cfg.hidden_size)

print(f"Model id              : {MODEL_ID}")
print(f"num_labels            : {model_cfg.num_labels}")
print(f"num_frames (T)        : {MODEL_T}     <- the model's expected temporal length")
print(f"image_size (H=W)      : {MODEL_H}    <- must match the loader's IMAGE_SIZE")
print(f"tubelet_size          : {TUBELET}    <- 3D conv kernel on the time axis")
print(f"patch_size            : {PATCH}")
print(f"hidden_size           : {HIDDEN}")
print(f"label2id              : {model_cfg.label2id}")
print(f"id2label              : {model_cfg.id2label}")

from pipeline_a import IMAGE_SIZE as LOADER_IMAGE_SIZE
assert LOADER_IMAGE_SIZE == MODEL_H, (
    f"Loader IMAGE_SIZE={LOADER_IMAGE_SIZE} != model image_size={MODEL_H}. "
    "The pipeline_a loader was pinned to MCG-NJU/videomae-base; verify the pinned model_id."
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"\nModel moved to device : {device}")

## 4. Torch Dataset wrapping `pipeline_a/load_crops.py`

Wraps the Issue 005 Step 1 loader in a `torch.utils.data.Dataset`. The wrapper does NOT re-normalise — the loader already emits ImageNet-normalised `(T, 3, H, W) float32` tensors, and `VideoMAEForVideoClassification` consumes them directly.

The wrapper handles the 32 → 16 temporal reconciliation via uniform subsampling, driven by the model's `num_frames` config attribute (NOT a hard-coded constant — if a future VideoMAE variant expects 32 frames, the same wrapper adapts).

Each `__init__` call also runs the **real-crop label** through `label_window_from_crop_meta(...)` (the only supported real-crop entry point per the Issue 006 carry-forwards). Loading + labelling happens once per clip and is cached on the instance — the tiny-slice overfit only touches a handful of clips.

In [ ]:
import numpy as np
from typing import Sequence

from pipeline_a import (
    ALLOWED_T,
    IMAGE_SIZE,
    LoadedClip,
    InvalidClipLengthError,
    load_clip_tensor_from_shards,
)
from data.urfd_labels import (
    CSVLabels,
    WINDOW_LABEL_FALL,
    WINDOW_LABEL_NO_FALL,
    WINDOW_LABEL_UNLABELED,
    label_window_from_crop_meta,
)

def uniform_temporal_subsample(frames: np.ndarray, target_T: int) -> np.ndarray:
    """Reconcile a clip's frame count to ``target_T`` via uniform subsampling.

    ``frames`` has shape ``(T, C, H, W)`` — the pipeline_a loader's contract.
    If ``T == target_T`` the array is returned unchanged. Otherwise we
    pick ``target_T`` evenly-spaced indices in ``[0, T - 1]``.
    """
    T = frames.shape[0]
    if T == target_T:
        return frames
    if T < target_T:
        raise ValueError(f"Cannot upsample {T} frames to {target_T}; the loader disallows pad/truncate.")
    if target_T == 1:
        return frames[:1]
    if (T - 1) % (target_T - 1) == 0:
        indices = np.arange(target_T) * ((T - 1) // (target_T - 1))
    else:
        indices = np.round(np.linspace(0.0, float(T - 1), target_T)).astype(np.int64)
    return frames[indices]

class VideoMAECropDataset(torch.utils.data.Dataset):
    """Wrap pipeline_a.load_crops.load_clip_tensor_from_shards as a torch Dataset.

    Each item is a dict with:
      - ``pixel_values``: torch.float32 tensor, shape ``(MODEL_T, 3, MODEL_H, MODEL_W)``,
        ImageNet-normalised by the loader (we do NOT re-normalise).
      - ``labels``: torch.long scalar with the window-level label
        (``0 = no_fall``, ``1 = fall``).
      - ``clip_key``, ``clip_id``: provenance, copied from the LoadedClip.
    """

    def __init__(
        self,
        clip_keys: Sequence[str],
        shard_paths: Sequence[object],
        csv_by_clip_id: dict[str, CSVLabels],
        target_T: int,
        target_H: int,
        target_W: int,
    ) -> None:
        self._clip_keys = list(clip_keys)
        self._shard_paths = list(shard_paths)
        self._csv_by_clip = dict(csv_by_clip_id)
        self._target_T = int(target_T)
        self._target_H = int(target_H)
        self._target_W = int(target_W)
        self._cache: dict[str, dict[str, object]] = {}

        skipped: list[tuple[str, str]] = []
        kept: list[str] = []
        for clip_key in self._clip_keys:
            try:
                entry = self._load_and_label(clip_key)
            except Exception as exc:  # noqa: BLE001
                skipped.append((clip_key, f"{type(exc).__name__}: {exc}"))
                continue
            if entry is None:
                skipped.append((clip_key, "unlabeled window"))
                continue
            self._cache[clip_key] = entry
            kept.append(clip_key)

        self._clip_keys = kept
        if not self._clip_keys:
            raise RuntimeError(
                "VideoMAECropDataset has no usable clips after labelling. "
                f"Skipped: {skipped}"
            )
        self._skipped = tuple(skipped)

    @property
    def skipped(self) -> tuple[tuple[str, str], ...]:
        return self._skipped

    def __len__(self) -> int:
        return len(self._clip_keys)

    def _load_and_label(self, clip_key: str) -> dict[str, object] | None:
        loaded: LoadedClip | None = None
        last_exc: Exception | None = None
        for T in ALLOWED_T:
            try:
                loaded = load_clip_tensor_from_shards(
                    self._shard_paths, clip_key=clip_key, T=T,
                )
                break
            except InvalidClipLengthError as exc:
                last_exc = exc
                continue
        if loaded is None:
            if last_exc is not None:
                raise last_exc
            raise RuntimeError(f"Could not load clip {clip_key!r} at any ALLOWED_T.")

        csv = self._csv_by_clip.get(loaded.clip_id)
        if csv is None:
            raise RuntimeError(
                f"No CSVLabels registered for clip_id {loaded.clip_id!r}. "
                f"Check that the CSV directory has both URFD label files."
            )

        label_str, is_confuser = label_window_from_crop_meta(
            loaded.clip_id, list(loaded.frame_indices), csv,
        )
        if label_str == WINDOW_LABEL_UNLABELED:
            return None
        if label_str == WINDOW_LABEL_FALL:
            label_int = 1
        elif label_str == WINDOW_LABEL_NO_FALL:
            label_int = 0
        else:
            raise RuntimeError(f"Unknown label_str {label_str!r} from label_window_from_crop_meta")

        tensor = uniform_temporal_subsample(loaded.tensor, self._target_T)
        if tensor.shape[1] != 3 or tensor.shape[2] != self._target_H or tensor.shape[3] != self._target_W:
            raise RuntimeError(
                f"Clip {clip_key!r}: tensor shape {tensor.shape} != expected "
                f"(T={self._target_T}, 3, {self._target_H}, {self._target_W}). "
                f"Pipeline A loader pinned at H=W={IMAGE_SIZE}; model expects {self._target_H}. "
                f"Mismatched pinning — verify the model id and the loader's constants."
            )

        return {
            "pixel_values": torch.from_numpy(tensor),
            "labels": torch.tensor(label_int, dtype=torch.long),
            "clip_key": clip_key,
            "clip_id": loaded.clip_id,
            "is_confuser": bool(is_confuser),
        }

    def __getitem__(self, idx: int) -> dict[str, object]:
        clip_key = self._clip_keys[idx]
        return self._cache[clip_key]

## 5. Real-crop labelling through `label_window_from_crop_meta`

Load the two URFD label CSVs once and build the `clip_id → CSVLabels` map the dataset uses. Every window's label is computed via `label_window_from_crop_meta(...)` — the supported real-crop entry point — and the primitive `label_window(...)` is never used here.

In [ ]:
from data.urfd_labels import parse_urfd_csv_label_file

fall_csv = parse_urfd_csv_label_file(CSV_LABEL_DIR / FALL_CSV_FILENAME)
adl_csv = parse_urfd_csv_label_file(CSV_LABEL_DIR / ADL_CSV_FILENAME)

print(f"Fall CSV sequences    : {len(fall_csv.sequences())}    (expected 30)")
print(f"ADL  CSV sequences    : {len(adl_csv.sequences())}    (expected 40)")

csv_by_clip_id: dict[str, CSVLabels] = {}
for clip_id in unique_clip_ids:
    if "fall-" in clip_id:
        csv_by_clip_id[clip_id] = fall_csv
    elif "adl-" in clip_id:
        csv_by_clip_id[clip_id] = adl_csv
    else:
        raise SystemExit(f"Unknown clip_id family: {clip_id!r}")

print(f"clip_id -> CSV map size : {len(csv_by_clip_id)}    (expected 70)")

## 6. Split by source clip (never by window)

Window-level splits leak across consecutive frames of the same fall — same person, same scene, same model-decision. Per the Issue 006 carry-forwards, the split must be at the **clip** level: every window of a clip lands in the same split.

We use a deterministic shuffled split (seed=42) with `train=70%`, `val=15%`, and the remainder held back. Frozen unseen tier is **inaccessible** from this scaffold — `ExecutionContext.EVALUATION` denies frozen access by design, so no `frozen_unseen_test` clips appear in any split.

In [ ]:
import random
from collections import defaultdict

SPLIT_SEED = 42
SPLIT_TRAIN = 0.70
SPLIT_VAL = 0.15

windows_by_clip: dict[str, list[tuple[str, str, str]]] = defaultdict(list)
for ck, ci, lbl in windows:
    windows_by_clip[ci].append((ck, ci, lbl))

all_clip_ids = sorted(windows_by_clip.keys())
rng = random.Random(SPLIT_SEED)
rng.shuffle(all_clip_ids)

n = len(all_clip_ids)
n_train = max(1, int(round(n * SPLIT_TRAIN)))
n_val = max(1, int(round(n * SPLIT_VAL)))
if n_train + n_val >= n:
    n_val = max(1, n - n_train - 1)

train_clip_ids = set(all_clip_ids[:n_train])
val_clip_ids = set(all_clip_ids[n_train:n_train + n_val])
held_clip_ids = set(all_clip_ids[n_train + n_val:])

def _windows_for(clips: set[str]) -> list[tuple[str, str, str]]:
    out: list[tuple[str, str, str]] = []
    for ci in clips:
        out.extend(windows_by_clip[ci])
    return out

train_windows = _windows_for(train_clip_ids)
val_windows = _windows_for(val_clip_ids)
held_windows = _windows_for(held_clip_ids)

# Debug-tier clip-id assertion — manifest-free. We deliberately
# avoid loading the URFD manifest here: in a standalone Issue 006
# session the manifest lives under layout.dataset_root / "datasets"
# / "urfd" / "manifest.yaml", which is on the Colab local disk
# and is wiped on runtime reset. A hard dependency on it would
# force every 006 re-run to re-stage URFD.
#
# Instead, we use the clip_ids already discovered from the crop
# shards in §2 — every Issue 003 crop shard carries the source
# clip_id in its metadata sidecar, so the debug-tier claim is
# recoverable from the shards alone. A clip_id that does NOT
# contain "-debug-" (e.g. a frozen-tier or non-debug clip_id)
# fails loud. Today's URFD debug-tier clips all match
# `urfd-debug-<seq>-cam0-rgb`; a future re-pointing at a vault
# dataset (omnifall / caucafall / mcfd / fallvision) would surface
# immediately.
all_crop_clip_ids = {r["clip_id"] for r in records}
non_debug = sorted(cid for cid in all_crop_clip_ids if "-debug-" not in cid)
assert not non_debug, (
    f"Debug-tier assertion failed: {len(non_debug)} clip_id(s) from the crop "
    f"shards do not look like debug-tier data. Sample: {non_debug[:3]}\n"
    f"This notebook reads only debug-tier crop shards produced by "
    f"colab/000_full_pipeline.ipynb. Frozen / non-debug / vault data must "
    f"NOT appear here — re-run Issue 003 with debug-tier source clips, "
    f"or point this notebook at a different CROPS_ROOT."
)

print(f"Clips total           : {n}")
print(f"  train clip ids      : {len(train_clip_ids)}")
print(f"  val   clip ids      : {len(val_clip_ids)}")
print(f"  held clip ids       : {len(held_clip_ids)}")
print(f"Windows total         : {len(windows)}")
print(f"  train windows       : {len(train_windows)}    (fall={sum(1 for w in train_windows if w[2]=='fall')})")
print(f"  val   windows       : {len(val_windows)}    (fall={sum(1 for w in val_windows if w[2]=='fall')})")
print(f"  held windows        : {len(held_windows)}    (fall={sum(1 for w in held_windows if w[2]=='fall')})")
print(f"Debug-tier assertion  : PASSED ({len(all_crop_clip_ids)} unique clip_ids, "
      f"all match `*-debug-*`)")

## 7. Tiny-slice overfit

Pick a handful of fall + no-fall windows from the train split, run a few AdamW steps, and assert that the model path is wired correctly. We deliberately avoid an **absolute** loss threshold because it depends on random initial weights and the label mix — a **relative** decrease is what actually proves the model is learning.

Pass criteria (both must hold; a failure is a wiring problem, not a model-capacity problem):

1. Every step's loss is **finite** (no NaN / Inf — usually an LR / dtype / label-mismatch symptom).
2. The final avg loss is at most **50%** of the initial avg loss (substantial decrease).

We pick **3 fall + 3 no-fall** windows so the overfit has enough signal to memorise without burning compute, and we run **5 steps** with lr=5e-5. The expected pattern: initial loss near `log(2) ≈ 0.69` for a 2-class head, final loss well under half of that within a few steps.

In [ ]:
import math

TINY_FALL = 3
TINY_NOFALL = 3
TINY_STEPS = 5
TINY_LR = 5e-5
TINY_LOSS_RATIO = 0.5   # final must be <= 50% of initial

tiny_falls = [w for w in train_windows if w[2] == "fall"][:TINY_FALL]
tiny_nofalls = [w for w in train_windows if w[2] == "no_fall"][:TINY_NOFALL]
tiny_slice = tiny_falls + tiny_nofalls
tiny_clip_keys = [w[0] for w in tiny_slice]

print(f"Tiny-slice clip keys  : {len(tiny_slice)}")
for w in tiny_slice:
    print(f"  {w[0]:<44}  clip_id={w[1]:<32}  metadata-label={w[2]}")

assert len(tiny_falls) >= 1, "Need at least one fall window for the overfit test."
assert len(tiny_nofalls) >= 1, "Need at least one no-fall window for the overfit test."

tiny_dataset = VideoMAECropDataset(
    clip_keys=tiny_clip_keys,
    shard_paths=shard_paths,
    csv_by_clip_id=csv_by_clip_id,
    target_T=MODEL_T,
    target_H=MODEL_H,
    target_W=MODEL_W,
)

print(f"\nDataset size          : {len(tiny_dataset)}")
print(f"Skipped at __init__   : {len(tiny_dataset.skipped)}")

probe = tiny_dataset[0]
print("\nProbe item:")
print(f"  pixel_values shape   : {probe['pixel_values'].shape}    (expected ({MODEL_T}, 3, {MODEL_H}, {MODEL_W}))")
print(f"  pixel_values dtype   : {probe['pixel_values'].dtype}    (expected torch.float32)")
print(f"  pixel_values range   : [{probe['pixel_values'].min():.3f}, {probe['pixel_values'].max():.3f}]    (ImageNet-normalised => ~[-2.1, 2.6])")
print(f"  labels               : {int(probe['labels'])}    (0=no_fall, 1=fall)")
print(f"  clip_id              : {probe['clip_id']}")

assert probe['pixel_values'].shape == (MODEL_T, 3, MODEL_H, MODEL_W), (
    f"Pixel-value shape {probe['pixel_values'].shape} != expected ({MODEL_T}, 3, {MODEL_H}, {MODEL_W})"
)

optimizer = torch.optim.AdamW(model.parameters(), lr=TINY_LR)
model.train()
step_losses: list[float] = []
for step in range(TINY_STEPS):
    per_step_losses: list[float] = []
    for i in range(len(tiny_dataset)):
        item = tiny_dataset[i]
        pixel_values = item["pixel_values"].unsqueeze(0).to(device)
        labels = item["labels"].unsqueeze(0).to(device)
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at step={step} item={i}: {loss.item()}")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        per_step_losses.append(float(loss.detach().cpu().item()))
    avg = sum(per_step_losses) / len(per_step_losses)
    step_losses.append(avg)
    print(f"  step {step+1}/{TINY_STEPS} avg_loss = {avg:.4f}")

final_loss = step_losses[-1]
initial_loss = step_losses[0]
print(f"\nInitial avg loss      : {initial_loss:.4f}")
print(f"Final   avg loss      : {final_loss:.4f}")

# Pass criteria:
#   1. every loss is finite (caught per-step above; torch.isfinite is
#      used on the per-step tensor loss inside the training loop)
#   2. final loss is at most TINY_LOSS_RATIO × initial loss
#
# initial_loss and final_loss are Python floats — torch.isfinite on a
# Python float raises TypeError. Use math.isfinite for the float check.
if not (math.isfinite(initial_loss) and math.isfinite(final_loss)):
    raise RuntimeError(
        f"Tiny-slice overfit FAILED: non-finite loss. "
        f"initial={initial_loss}, final={final_loss}. "
        f"This is a WIRING problem (NaN / Inf usually = LR too high, "
        f"label mismatch, or dtype issue)."
    )
if final_loss > TINY_LOSS_RATIO * max(initial_loss, 1e-9):
    raise RuntimeError(
        f"Tiny-slice overfit FAILED: final avg loss {final_loss:.4f} did not "
        f"decrease substantially from initial {initial_loss:.4f} "
        f"(ratio {final_loss / max(initial_loss, 1e-9):.2f} > threshold {TINY_LOSS_RATIO:.2f}). "
        f"This is a WIRING problem (data pipeline / labels / head / optimiser / "
        f"tensor shape), not a model-capacity problem. Inspect the per-step losses above."
    )
print("Tiny-slice overfit    : PASSED (loss is finite and decreased substantially).")

## 8. Evaluation-harness plumbing

Send the same tiny-slice predictions through `evaluation.metrics.classification.compute_classification_metrics` and print **AUPRC** + **recall** (the priority metrics for rare fall detection per the PRD — accuracy is intentionally not surfaced because class imbalance makes it deceptively high).

These numbers are **not formal results**. The harness plumbing is the deliverable; the formal Pipeline A results land in Issue 007.

In [ ]:
from data.manifests import ClipRole, FallLabel
from evaluation.contracts import ClipLabel, ClipPrediction
from evaluation.metrics.classification import compute_classification_metrics
from evaluation.not_available import NotAvailable

model.eval()
predictions: list[ClipPrediction] = []
labels_list: list[ClipLabel] = []

with torch.no_grad():
    for i in range(len(tiny_dataset)):
        item = tiny_dataset[i]
        pixel_values = item["pixel_values"].unsqueeze(0).to(device)
        outputs = model(pixel_values=pixel_values)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)[0].detach().cpu().numpy()
        fall_score = float(probs[1])
        clip_key = str(item["clip_key"])
        clip_id = str(item["clip_id"])
        label_int = int(item["labels"])
        predictions.append(ClipPrediction(
            clip_id=clip_key,
            score=fall_score,
            model_id="MCG-NJU/videomae-base-tiny-slice",
            dataset="urfd",
            role=ClipRole.TRAIN,
        ))
        labels_list.append(ClipLabel(
            clip_id=clip_key,
            label=FallLabel.FALL if label_int == 1 else FallLabel.NO_FALL,
            dataset="urfd",
            role=ClipRole.TRAIN,
            source_path=f"shard://{clip_id}",
        ))

reports = compute_classification_metrics(predictions, labels_list)

print(f"Classification reports : {len(reports)}    (1 aggregate + N per-slice)")
print()

for report in reports:
    if report.slice_key is None:
        print("Aggregate metrics (eval-harness plumbing smoke):")
        for metric in report.metrics:
            if metric.name not in ("recall", "auprc", "accuracy", "precision", "f1", "auc_roc"):
                continue
            v = metric.value
            if isinstance(v, NotAvailable):
                print(f"  {metric.name:<10} : n/a ({v.reason})")
            else:
                print(f"  {metric.name:<10} : {v:.4f}")
        break

print()
print("Per-slice reports (dataset, action_confuser, etc.):")
for report in reports:
    if report.slice_key is None:
        continue
    n_pos = report.support.n_positive
    n_neg = report.support.n_negative
    print(f"  {report.slice_key.tag}={report.slice_key.value:<12}  n_pos={n_pos:>3}  n_neg={n_neg:>3}")
    for metric in report.metrics:
        if metric.name in ("recall", "auprc"):
            v = metric.value
            if isinstance(v, NotAvailable):
                print(f"      {metric.name:<10} : n/a ({v.reason})")
            else:
                print(f"      {metric.name:<10} : {v:.4f}")

## 9. Done — Issue 006 Prompt #1 scope

What this scaffold proved:

- Crop shards exist on Drive with the expected shape (~70 shards, ~240 windows, 41 fall, 199 no-fall). Loud guard fails if not.
- VideoMAE-base loads with a binary head and reports its expected frame count + image size.
- The pipeline_a loader produces tensors of the right shape (`(MODEL_T, 3, MODEL_H, MODEL_W)`), ImageNet-normalised, ready to feed the model.
- 32 → 16 frame reconciliation via uniform temporal subsampling adapts to the model's `num_frames` config (NOT a hard-coded constant).
- Real-crop labels go through `label_window_from_crop_meta(...)` only — primitive `label_window(...)` is never called on real crop data.
- Split is by source clip, never by window. Frozen unseen tier is inaccessible (EVALUATION denies it).
- Tiny-slice overfit collapses the loss below the wiring-threshold, proving the wiring.
- Eval-harness plumbing produces AUPRC + recall rows for the aggregate (and slices if any are present).

What is **not** in this prompt and is **deferred**:

- Full fine-tune run + scheduler + LR warmup + early stopping.
- Real validation + frozen-unseen test (out of scope at this step).
- Checkpoint / resume (Issue 006 spec calls it out — separate prompt).
- Class-imbalance handling (weighted loss / oversampling). The full 41-vs-199 imbalance is addressed in the next prompt.
- Pipeline C fusion / post-verification (Issues 011–012).

Outputs on Drive (artefact root only):

- `metrics/pipeline_a/006_pipeline_a_run_log.json` — env record (GPU, VRAM, CUDA, torch, bf16/fp16).
- **No model weights or checkpoints committed.** The `checkpoints/pipeline_a/` directory exists but is empty at this prompt.

If you want to extend this: bump `TINY_FALL` / `TINY_NOFALL` to a real validation slice, swap the optimizer for a real schedule, or wire `Trainer` / `TrainingArguments` — the wiring is in place to support any of those.